# Level OLS

Fig. 2(c). Relative exposure on relative supply and relative engagement, in levels. Breusch-Godfrey, Breusch-Pagan, and the ARDL(2, 2) follow-up are in `analysis/robustness/timeseries/ols.ipynb`.

### 1. The relative series

S_t is `rel_supply`, the labeled-to-unlabeled upload ratio. E_t is `rel_engagement`, the ratio of mean valid-view rates. R_t is `rel_exposure`, the ratio of 31-day median exposures. The regression enters S_t and E_t separately. It does not impose the equal weights in SoEI_t = ln(S_t / E_t).

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

import pandas as pd
from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
daily = ensure_synthetic(root, cfg, rebuild=False)["daily_panel"]
relative = daily[["rel_supply", "rel_engagement", "rel_exposure"]]
print(len(relative), "days")
relative.head(8)


180 days


,rel_supply,rel_engagement,rel_exposure
0,1.350000,0.820000,1.124531
1,1.367890,0.813994,1.077343
2,1.333979,0.801790,1.003102
3,1.348902,0.806906,1.021435
4,1.352439,0.794890,1.023771
5,1.338880,0.794388,0.999757
6,1.351665,0.810942,1.068481
7,1.373910,0.806425,1.034614


### 2. Newey-West coefficients

R_t = alpha + beta_S S_t + beta_E E_t + lambda D_t + epsilon_t. The bandwidth is 7 days. These two coefficients are the slopes drawn in Fig. 2(c).

In [2]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import run_ols_notebook

table = run_ols_notebook()
table[["term", "estimate", "standard_error", "p_value", "ci95_low", "ci95_high"]]


,term,estimate,standard_error,p_value,ci95_low,ci95_high
0,rel_supply,-0.860810,0.053098,4.159547e-59,-0.964879,-0.756740
1,rel_engagement,1.510424,0.116006,9.396002e-39,1.283055,1.737792


### 3. Partial residuals

Each panel residualizes R_t and one regressor on the other regressor and on D_t. The line is the Newey-West coefficient from the previous cell, not a new fit through the scatter.

In [3]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

partial = pd.read_csv(root / "data" / "results" / "level_ols_partial.csv")
print(table.set_index("term")["estimate"].to_string())
partial.groupby("regressor", sort=False).head(4)


term
rel_supply       -0.860810
rel_engagement    1.510424


,regressor,residual_regressor,residual_outcome
0,rel_supply,-0.010531,0.086796
1,rel_supply,0.011331,0.041307
2,rel_supply,-0.016344,-0.021638
3,rel_supply,-0.008680,0.010643
180,rel_engagement,0.009744,0.092448
181,rel_engagement,0.004527,0.057898
182,rel_engagement,-0.003031,-0.040285
183,rel_engagement,-0.005092,-0.004519
